# Session 2 — GPU: XGBoost + LightGBM + MLP + KAN + Save

**Runtime: T4 GPU หรือ H100 GPU** 

> ต้องรัน `01_CPU_Train_BRF.ipynb` ให้เสร็จก่อน เพื่อสร้าง cache ใน Drive

Steps:
1. Mount Google Drive + ดึง cache
2. Clone / pull latest code
3. Install dependencies
4. Load secrets
5. ตรวจสอบ GPU
6. Train XGBoost
7. Train LightGBM
8. Train MLP (Neural Network)
9. Train KAN
10. Save + Upload HuggingFace + Redeploy Render

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## 2. Clone / pull latest code

In [ ]:
import os

REPO_URL  = 'https://github.com/MCTEEKUNG/Heatwave_Backend_Elysia.git'
REPO_DIR  = '/content/Heatwave-AI'
BRANCH    = 'feat/eda-wbgt-year-split'
TRAIN_DIR = f'{REPO_DIR}/Heatwave-AI-TRAIN'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(TRAIN_DIR)
print(f'Working directory: {os.getcwd()}')
!git log --oneline -3

## 3. Install dependencies

In [ ]:
!pip install -r requirements.txt -q
print('Dependencies installed.')

## 4. Load secrets

In [ ]:
from google.colab import userdata
import os

os.environ['HF_TOKEN']           = userdata.get('HF_TOKEN')
os.environ['RENDER_API_KEY']     = userdata.get('RENDER_API_KEY')
os.environ['RENDER_SERVICE_ID']  = userdata.get('RENDER_SERVICE_ID')

!hf auth login --token $HF_TOKEN
print('Secrets loaded.')

## 5. ตรวจสอบ GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {vram_gb:.1f} GB')
else:
    print('ไม่พบ GPU!')
    print('ไปที่ Runtime > Change runtime type > GPU แล้วรันใหม่')
    raise SystemExit('GPU required')

## 6. ตรวจสอบ Cache จาก Session 1

ถ้า cache ไม่พบ ให้กลับไปรัน `01_CPU_Train_BRF.ipynb` ก่อน

In [ ]:
import os

CACHE_DIR = '/content/drive/MyDrive/Heatwave-AI/cache'
required  = ['X_train.npy', 'X_val.npy', 'X_test.npy',
              'y_train.npy', 'y_val.npy', 'y_test.npy',
              'scaler.pkl', 'imputer.pkl', 'feature_names.pkl', 'meta.json']

missing = [f for f in required if not os.path.exists(f'{CACHE_DIR}/{f}')]
if missing:
    print(f'Cache ไม่ครบ: {missing}')
    print('กลับไปรัน 01_CPU_Train_BRF.ipynb ก่อน')
    raise SystemExit('Cache missing')

import json, numpy as np
with open(f'{CACHE_DIR}/meta.json') as f:
    meta = json.load(f)

print('Cache พบครบ!')
print(f"  Features : {meta['feature_names']}")
print(f"  Train    : {meta['n_train']:,} rows  ({meta['heatwave_rate_train']:.2%} heatwave)")
print(f"  Val      : {meta['n_val']:,} rows")
print(f"  Test     : {meta['n_test']:,} rows  ({meta['heatwave_rate_test']:.2%} heatwave)")

## 7. Train XGBoost

รองรับ GPU (`tree_method=hist`, `device=cuda`) — ใช้เวลา ~2–5 นาที

In [ ]:
!python train_split.py --phase gpu --model xgboost --config config/config.yaml

## 8. Train LightGBM

รองรับ GPU (`device=gpu`) — ใช้เวลา ~2–5 นาที

In [ ]:
!python train_split.py --phase gpu --model lightgbm --config config/config.yaml

## 9. Train MLP (Neural Network)

PyTorch บน GPU — ใช้เวลา ~5–15 นาที (100 epochs + early stopping)

In [ ]:
!python train_split.py --phase gpu --model mlp --config config/config.yaml

## 10. Train KAN (Kolmogorov-Arnold Network)

PyTorch บน GPU — ใช้เวลา ~5–10 นาที (50 epochs + early stopping)

In [ ]:
!python train_split.py --phase gpu --model kan --config config/config.yaml

## 11. ดู Leaderboard — เปรียบเทียบทุกโมเดล

In [ ]:
import json, os

lb_path = 'experiments/results/leaderboard.json'
if os.path.exists(lb_path):
    with open(lb_path) as f:
        lb = json.load(f)

    print(f"{'Model':<22} {'F1':>8} {'Precision':>10} {'Recall':>8} {'ROC-AUC':>8}")
    print('-' * 60)
    for entry in sorted(lb, key=lambda x: x.get('f1_score', 0), reverse=True):
        print(f"{entry.get('model','?'):<22} "
              f"{entry.get('f1_score',0):>8.4f} "
              f"{entry.get('precision',0):>10.4f} "
              f"{entry.get('recall',0):>8.4f} "
              f"{entry.get('roc_auc',0):>8.4f}")
else:
    print('leaderboard.json ยังไม่มี — รัน train ก่อน')

## 12. Save → Upload HuggingFace → Redeploy Render

ขั้นตอนนี้จะ:
1. คัดลอก `.pkl` ทั้งหมดไปยัง Google Drive
2. Upload ขึ้น HuggingFace Hub (`MCTEEKUNG123/heatwave-ai-models`)
3. Trigger Render redeploy อัตโนมัติ (backend จะ pull โมเดลใหม่ใน ~10 นาที)

In [ ]:
!python train_split.py --phase save --config config/config.yaml

## เสร็จสมบูรณ์!

ทั้ง 5 โมเดลถูก train และ deploy แล้ว:

| โมเดล | Runtime | เหมาะกับ |
|-------|---------|----------|
| Balanced Random Forest | CPU | Baseline, interpretable |
| XGBoost | GPU | Tabular data champion |
| LightGBM | GPU | เร็ว, ประหยัด memory |
| MLP | GPU | Neural network baseline |
| KAN | GPU | Interpretable neural network |

Backend บน Render จะ reload โมเดลใหม่อัตโนมัติภายใน 10 นาที

URL: https://heatwave-backend-elysia.onrender.com